# Counting barnacles with zero-shot Cellpose-SAM

DALI Lab challenge, Fall 2026.

**Contents**
0. Summary
1. How to reproduce these results
2. Understanding the problem
3. Choosing a model
4. Ground truth: from hand-drawn outlines to labels
5. Restricting the image to the quadrat
6. Zero-shot baseline
7. Giving the model a barnacle diameter (main result)
8. Preprocessing experiments
9. The unseen images
10. Conclusions and next steps

## 0. Summary

| image | ground truth | cpsam, zero-shot + diameter | count error | F1 |
|---|---|---|---|---|
| img1 | 1694 | 1884 (d = 30) | +11.2 % | 0.926 |
| img2 | 396 | 403 (d = 50) | +1.8 % | 0.849 |
| unseen_img1 | – | 1377 (d = 30) | – | – |
| unseen_img2 | – | 157 (d = 30) | – | – |

I've applied a cell-counting model to this problem without fine-tuning and it achieved POC-level F1-scores and recall rates. In the later stages I have also experimented with various methods of preprocessing, not leading to better performance. To elevate this model from POC, fine-tuning on the example data is the feasible next step, after that lie other optimization steps depending on the needs of the researchers as well.

In [ ]:
import sys
sys.path.insert(0, "scripts")  # the helper modules live in scripts/, next to the pipeline scripts

from notebook_helpers import load_metrics, results_table, show_overlay, compare_overlays

metrics = load_metrics()  # processed_data/cellpose_zeroshot/metrics.json, one row per run

## 1. How to reproduce these results

This notebook does not run any models. It only reads files the pipeline already wrote
(`metrics.json`, the overlays, and the QC images), so it runs in seconds and shows no Cellpose log output.
Everything shown here can be regenerated from `raw_data/` with one command:

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt        # pinned: cellpose 4.2.1.1, numpy, scipy, scikit-image, ...
pip install ipykernel                  # only needed to run this notebook
bash reproduce.sh                      # about 15 min on an Apple M4 (MPS); a CUDA GPU or CPU also works
```

The first run of `run_cellpose_zeroshot.py` downloads the `cpsam` and `cpsam_v2` weights
(to `~/.cellpose/models`), so it needs internet access once.

### What `reproduce.sh` runs

| step | command | reads | writes |
|---|---|---|---|
| 1. labels | `python scripts/make_annotations.py` | `raw_data/mask{1,2}.png` | `processed_data/img{1,2}_masks.tif`, `processed_data/qc/` |
| 2. quadrat crop | `python scripts/crop_quadrat.py` | `raw_data/*.png`, the labels from step 1 | `processed_data/quadrat/` (crops, quadrat masks, cropped labels, `quadrat_boxes.json`) |
| 3. baseline | `run_cellpose_zeroshot.py --group first_Zeroshot` | `processed_data/quadrat/` | `processed_data/cellpose_zeroshot/{cpsam,cpsam_v2}/`, `metrics.json` |
| 4. diameter | `... --images img1 img2 --diameter img1=30 img2=50 --group Zeroshot_with_Diameter` | | |
| 5. preprocessing | six runs, one per group in Section 8 (exact flags in `reproduce.sh`) | | also `processed_data/cellpose_zeroshot/inputs/` |
| 6. deliverables | copy the four cpsam diameter overlays | | `results/` |

Step 2 must come after step 1, because it crops the label images as well as the photos.
Steps 3–5 append to `metrics.json` and do not depend on each other. Rerunning a group with the same settings
replaces its rows instead of duplicating them.

### Rerunning a single experiment

Each group in `metrics.json` is one call of the runner. For example, the main result (Section 7):

```bash
python scripts/run_cellpose_zeroshot.py --images img1 img2 --diameter img1=30 img2=50 --group Zeroshot_with_Diameter
```

Output file names encode every non-default setting (`img2_d50_claheL_overlay.png`, `img2_d50_p1-99.5_tn150_overlay.png`, ...),
so runs with different settings never overwrite each other.
`python scripts/run_cellpose_zeroshot.py --help` lists all options.

To check a rerun against the numbers in this notebook without overwriting them:
`CELLPOSE_OUT=rerun_check bash reproduce.sh` writes predictions and a fresh `metrics.json` to `rerun_check/` instead.

### How the scoring works (needed to read the tables)

* **Count error**: (predicted − annotated) / annotated. This is the number a researcher cares about.
* **Precision / recall / F1** use a *coverage match*: an annotation counts as found if one prediction covers ≥ 50 % of its pixels,
  matched one-to-one (Hungarian algorithm), so one prediction covering three barnacles counts once.
  This is needed because the annotations mark the barnacle **opening** while Cellpose outlines the **whole shell**,
  so the standard IoU ≥ 0.5 matching would reject almost every correct detection (AP50 ≈ 0.01 on img1).
* In the overlays, **red** = human annotation and **yellow** = Cellpose prediction. A yellow ring with a red outline inside it is a correct detection.

### Notes on exactness

* Cellpose runs in bfloat16 by default. On a different GPU or on CPU, counts can differ by a few barnacles.
  `--fp32` and `--cpu` are available in the runner.
* The `Composite_tilenorm` group was run before `--composite-channels` existed, with the channels then hard-coded as
  `clahe_l inv_sat dog`. `reproduce.sh` reproduces it with that flag. The output files are now named `..._comp-clahe_l-inv_sat-dog_...`
  instead of `..._composite_...`.
* Checked on 2026-10-21 by running `reproduce.sh` into a separate folder: the labels and quadrat crops matched exactly, and so did
  34 of the 38 rows in `metrics.json`. The 4 `CLAHE_L_gray` rows were off by 1–3 barnacles (F1 within ±0.001),
  most likely because `preprocessing.py` was changed after that run.
* Library versions: Python 3.13, see `requirements.txt`.

## 2. Understanding the problem

**Task:** count the live barnacles inside the inner quadrat of the wire grid, and mark them on the image so a researcher can check the count.

> The model needs to count the barnacles inside the inner square, reliably find labels that researchers could look over, and overall produce a count that is as close as possible to the real count; qualification: I am assuming that the researchers can spend some time looking over the results, which means that I prioritize higher recall such that deleting false counts is the work rather than drawing new ones.

## 3. Choosing a model

Bounding-Box detectors are weaker for counting densely packed objects, which is where other models specific to segmentation shine better. cellpose-SAM uses SAM as a base model and has trained it specifically to count cells from biological imaging of cells or similar. It generalizes well to different types of cells and also generally improves quickly via fine-tuning, which made it a great candidate for this problem.

## 4. Ground truth: from hand-drawn outlines to labels

`scripts/make_annotations.py` turns the hand-drawn outline masks into integer label images (one ID per barnacle),
which is what both scoring and any later fine-tuning need.
Result: **1694 barnacles in img1, 396 in img2.**

Things I found by inspection:
* The masks outline the barnacle **opening**, not the whole shell. Some openings are only a few pixels across (small barnacles, or ones seen side-on on a ridge).
* The brush width differs a lot between the two masks (≈ 2.6 px for mask1, ≈ 6.3 px for mask2).

Decisions: interiors smaller than 10 px are dropped (brush artefacts), and each interior is grown outwards by half the local brush width so the label edge sits on the stroke centreline.

In [ ]:
show_overlay("processed_data/qc/img1_qc_overlay.png", "img1: derived labels on the photo", zoom=(1000, 1500, 300));

> Generally, these annotations as the ground truth are accurate and correctly count the barnacles. Some simplifications were made, like the minimum inside amount of pixels for a barnacle to count, which was necessary when a boundary failed to mark a single pixel within the boundary for example. At the same time, there were some barnacles likely to be missed that way. Also, there are still some errors in the ground truth which make the system more fragile (see the missed barnacle in the bottom left of the zoomed in photo: zooming in on that example in the provided mask shows a not closed loop.)

## 5. Restricting the image to the quadrat

Only barnacles inside the large central cell of the grid are annotated, so the model has to look at that cell only,
or everything outside would count as a false positive. `scripts/crop_quadrat.py` finds the teal wire by colour (HSV threshold),
takes the largest cell fully enclosed by wire, and crops to it. Predictions whose centre falls outside the cell are discarded.
`unseen_img2` is a 500 × 500 close-up without a grid, so it is used whole.

In [ ]:
show_overlay("processed_data/quadrat/img2_qc.png", "img2: detected quadrat (red)");

> the grid is consistently used again and has a very distinct color hue, which allows for simpler systems to very effectively and accurately detect the grid. This simple approach thus becomes very efficient to crop to the required area.

## 6. Zero-shot baseline

Both models with their pretrained weights and every Cellpose default. No setting is tuned against our images.

In [ ]:
results_table(metrics, ["first_Zeroshot"])

* img1: recall ≈ 0.98, F1 ≈ 0.92, but about 11 % overcount. Many "false positives" are empty or dead shells, which the annotator skipped.
* img2 (darker, lower contrast): recall only 0.21–0.25, even for `cpsam_v2`, which was updated for low contrast.

> img2 has lower contrast, and the size of the barnacles overall is different: they seem harder to spot and the model reflects that. These are important attributes to take away with, as we will try to improve on these mainly; the different size of the barnacles can be directly addressed!

## 7. Giving the model a barnacle diameter (main result)

Cellpose rescales the image so that objects are about 30 px across, given an expected `diameter`.
I estimated the diameter by eye, the way a researcher using the tool would: **30 px for img1, 50 px for img2.**
For img1, 30 px is the model's own reference size, so nothing changes there. For img2 the effect is large:

In [ ]:
results_table(metrics, ["first_Zeroshot", "Zeroshot_with_Diameter"])

* img2 with `cpsam`: F1 0.34 → **0.85**, count **403 vs 396 (+1.8 %)**.
* `cpsam` now beats `cpsam_v2` (F1 0.77, −24.5 %), which misses many small dark openings.

### img1, d = 30 (cpsam)

In [ ]:
show_overlay("results/img1_d30_overlay.png", "img1, cpsam, d = 30: 1884 found vs 1694 annotated", zoom=(150, 950, 300));

### img2, d = 50 (cpsam)

In [ ]:
show_overlay("results/img2_d50_overlay.png", "img2, cpsam, d = 50: 403 found vs 396 annotated", zoom=(1000, 1050, 400));

### img2 before and after setting the diameter (same window)

In [ ]:
compare_overlays(
    ["processed_data/cellpose_zeroshot/cpsam/img2_overlay.png", "results/img2_d50_overlay.png"],
    ["cpsam, no diameter: 95 found", "cpsam, d = 50: 403 found"],
    zoom=(1000, 1050, 400),
);

> The model is described as robust to size differences, but still performs much better when the barnacles are of the most trained size. Specifying the size hear significantly improved the results.
What contributes to the count are also the false negatives: empty and dead shells are hard to differentiate without fine-tuning on this specific data. Together with the missed actual barnacles, the resulting count can be very close to the actual number without necessarily representing very accurate counting. Fine-Tuning and Preprocessing might improve on these results.

## 8. Preprocessing experiments

All runs use the diameters from Section 7. Code: `scripts/preprocessing.py`. Cellpose-SAM takes 3 input channels in any order,
so I tried both contrast enhancement and "composites" that put a different cue in each channel.

| group | what the model sees |
|---|---|
| `RGB_norm_p1-99.5_tile150` | plain RGB; Cellpose normalisation computed per 150-px tile, percentiles 1–99.5 |
| `CLAHE_L_gray` | CLAHE on the CIELAB lightness channel, copied into all 3 channels |
| `Composite_tilenorm` | CLAHE-L / inverted HSV saturation / difference of Gaussians, + tile normalisation |
| `Composite_DoG_CLAHE_gray_tilenorm` | DoG / CLAHE-L / grayscale, + tile normalisation |
| `Composite_CLAHEx3_tilenorm` | CLAHE-L in all 3 channels, + tile normalisation (ablation) |
| `Composite_CLAHEx2_DoG_tilenorm` | CLAHE-L / CLAHE-L / DoG, + tile normalisation (ablation) |

In [ ]:
results_table(metrics, [g for g in metrics["group"].cat.categories if g != "first_Zeroshot"])

* CLAHE-L raises recall on img2 to ≈ 0.98 (best F1, 0.884) but overcounts by 21–30 %: it trades precision for recall.
* Tile normalisation alone gives a small gain; on top of CLAHE it changes nothing.
* Any composite containing a DoG or saturation channel collapses on img2. The ablations isolate the DoG channel as the cause.

In [ ]:
compare_overlays(
    ["results/img2_d50_overlay.png", "processed_data/cellpose_zeroshot/cpsam/img2_d50_claheL_overlay.png"],
    ["RGB, d = 50", "CLAHE-L, d = 50"],
    zoom=(1000, 1050, 400),
);

> Looking into preprocessing, providing the zeroshot model generally with data in a layout it is unfamiliar with caused the model to collapse to much worse performances. One notable find is that if all channels (the model can receive 3: RGB for example takes up all 3) are the clahe-L, so lumination-contrast-heightened, the recall of the model rises significantly. The model is much less likely to miss barnacles, but also thus is much more likely to count empty shells as living barnacles; overall, I argue this behavior is preferable to missed barnacles, since researchers can have an easier time deleting wrong labels than drawing new ones. Thus, both RGB and clahe-L are processing approaches that show promise to yield a functioning model after fine-tuning.

This is also likely the largest lesson I take from this project as various attempts at filtering, like for edges, for saturation inverses, yielded such large drops in performance, that it demonstrates the importance of aligning a model's input data to what it was trained on, even if what the inputs depict is different: input format can be very fragile.

## 9. The unseen images

No ground truth exists for these, so only the count and the overlay can be judged. Both use `cpsam`, d = 30.

In [ ]:
results_table(metrics, ["Zeroshot_with_Diameter"], images=["unseen_img1", "unseen_img2"]).hide(
    ["n_gt", "count_error_pct", "precision", "recall", "f1"], axis="columns"
)

In [ ]:
show_overlay("results/unseen_img1_d30_overlay.png", "unseen_img1, cpsam, d = 30: 1377 found", zoom=(600, 600, 300));
show_overlay("results/unseen_img2_d30_overlay.png", "unseen_img2, cpsam, d = 30: 157 found (no quadrat, whole image)");

> When we apply this approach to the unseen images, we receive similar results. Here, the model benefits from the unseen image's closeness to image 1, which it clarifies with much better performance overall than image 2. Also visible as expected, are the classifications of dead/empty barnacles as false positives. This is a problem that fine-tuning would need to address and is not unlikely to find success with.

## 10. Conclusions and next steps

> Using modern segmentation models and those derived from them can yield fast and useful solutions to problems like this one. It demonstrates the power of generalizing models making jumps from cell-counting to barnacles, real-life objects millions of times larger but similar on a photograph. It also then shows the importance of fine-tuning for problem specific circumstances, like the dead empty barnacles. Also, a significant improvement was giving the general size of the barnacles as was read via eye and pixel-measuring tool by a human; this is a step whose automation should be examined, but it also shows another spot where a short, for humans simple task can improve outcomes greatly. Next steps after this then are fine-tuning to see the model's performance then, and after then general optimization: collecting more data to fine-tune and evaluate on, optimizing the preprocessing further, also customized depending on the input image, and taking more steps into making the model as reliant as possible without human input needed.
> I got the opportunity to also look further into image processing with this project: I studied the segmentation model's and their application, and was quite surprised to find how quickly their performance can vanish with altered inputs that I thought would aid due to the new pieces of information. It shows how important it is to pay close attention to what inputs exactly a model expects or was trained and performs best on.